# 02 — Duchenne Smile Detection

A **Duchenne smile** is a smile that activates both the mouth (AU12, *zygomaticus major*) **and** the outer-eye muscles (AU06, *orbicularis oculi* — cheek raiser / crow's feet). Duchenne smiles are associated with genuine felt positive emotion; AU12-only smiles (non-Duchenne, a.k.a. "social" or "polite" smiles) can be produced voluntarily and occur in polite/masking contexts.

For trust research this distinction matters because the **ratio** of Duchenne to non-Duchenne smiles is a stronger predictor of perceived warmth/trustworthiness than total smile time alone.

**Input:** `data/<video>_merged.parquet` from `00_pipeline.ipynb`.  
**Outputs:** per-event table, overall counts, Duchenne ratio, overlaid AU12/AU06 plot with event shading, optional saved event parquet.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

## Config

Set `VIDEO_STEM` to your video's filename stem. Thresholds and minimum event duration are tunable — defaults chosen for Py-Feat 0.6.2 (AU scale is 0–1, not the historical 0–5).

In [ ]:
VIDEO_STEM = "sample"

AU12_THRESHOLD = 0.5     # AU12 active → smile candidate
AU06_THRESHOLD = 0.5     # AU06 active during the smile → Duchenne
MIN_EVENT_FRAMES = 2     # drop flash-in-the-pan events shorter than this
MERGE_GAP_FRAMES = 1     # merge events separated by ≤ this many below-threshold frames

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
PARQUET_PATH = DATA_DIR / f"{VIDEO_STEM}_merged.parquet"
META_PATH = DATA_DIR / f"{VIDEO_STEM}_merged.meta.json"
EVENTS_OUT = DATA_DIR / f"{VIDEO_STEM}_duchenne_events.parquet"

print("Input :", PARQUET_PATH)
print("Output:", EVENTS_OUT)

In [ ]:
df = pd.read_parquet(PARQUET_PATH)
meta = pd.read_json(META_PATH, typ="series")
FPS = float(meta["effective_fps"])
frame_duration_s = 1.0 / FPS
print(f"{len(df)} sampled frames, effective fps = {FPS:.2f}")
df[["frame", "timestamp", "pf_AU12", "pf_AU06"]].head(3)

## 1. Detect smile events

**What this section does:** converts the continuous AU12 intensity signal into a discrete list of "smile events" — stretches of consecutive frames where AU12 was above threshold.

**How the event detection works:**
1. Mark each frame as "on" if `pf_AU12 > AU12_THRESHOLD`, else "off".
2. Find contiguous runs of "on" frames.
3. Merge runs separated by ≤ `MERGE_GAP_FRAMES` brief "off" frames — smooths over single-frame detection dropouts mid-smile.
4. Drop events shorter than `MIN_EVENT_FRAMES` (below ~0.2s at 6 fps) — these are likely noise.

**Why these parameters:**
- `AU12_THRESHOLD = 0.5` — Py-Feat AU intensities are bimodal (either ~0 or ~0.9 when active), so any threshold between 0.3 and 0.7 gives similar results. 0.5 is the clean midpoint.
- `MERGE_GAP_FRAMES = 1` — detection can blip off for a single frame on a fast head turn or blink. Merging one-frame gaps avoids fragmenting what is clearly one smile.
- `MIN_EVENT_FRAMES = 2` — at 6 effective fps this means smiles must last at least ~0.3s to count. Real smiles are typically 0.5–4s (Ekman & Friesen, 1982).

In [ ]:
def find_events(mask: np.ndarray, merge_gap: int = 0, min_length: int = 1):
    """Return list of (start_idx, end_idx_inclusive) for runs of True in mask.
    Merges runs separated by <= merge_gap False entries, then drops runs < min_length."""
    mask = np.asarray(mask).astype(bool)
    if mask.size == 0:
        return []
    diff = np.diff(mask.astype(int), prepend=0, append=0)
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0] - 1  # inclusive
    events = list(zip(starts.tolist(), ends.tolist()))

    if merge_gap > 0 and len(events) > 1:
        merged = [events[0]]
        for s, e in events[1:]:
            ps, pe = merged[-1]
            if s - pe - 1 <= merge_gap:
                merged[-1] = (ps, e)
            else:
                merged.append((s, e))
        events = merged

    events = [(s, e) for s, e in events if (e - s + 1) >= min_length]
    return events

au12 = df["pf_AU12"].to_numpy()
au06 = df["pf_AU06"].to_numpy()
timestamps = df["timestamp"].to_numpy()
frames = df["frame"].to_numpy()

smile_mask = au12 > AU12_THRESHOLD
event_ranges = find_events(smile_mask, merge_gap=MERGE_GAP_FRAMES, min_length=MIN_EVENT_FRAMES)
print(f"Detected {len(event_ranges)} smile events.")

## 2. Classify each event as Duchenne or non-Duchenne

**What this section does:** for each smile event, checks whether AU06 (cheek raiser) was also active during the event and labels it accordingly.

**Classification rule:** `type = "duchenne"` if the **peak** AU06 within the event exceeds `AU06_THRESHOLD`, else `"non-duchenne"`.

**Why peak (not mean):** a smile might activate AU06 only at its apex (e.g. a smile that briefly spikes into a genuine grin). Taking the peak catches that. Mean AU06 would dilute the signal across the whole event duration.

**Why AU06:** it's the single AU that distinguishes Duchenne from non-Duchenne in Ekman's original work. AU06 is produced by *orbicularis oculi pars orbitalis*, which is difficult (though not impossible) to activate voluntarily — making it a reasonable biomarker for felt enjoyment.

**Caveats:**
- Some people *can* voluntarily activate AU06. The Duchenne marker is a statistical tendency, not a lie-detector.
- Big smiles naturally raise the cheeks as a mechanical side effect — very intense non-Duchenne smiles can incidentally trigger AU06. Recent literature (e.g. Krumhuber & Manstead, 2009) flags this. For high-stakes claims, consider an additional intensity/duration filter on AU06.

In [ ]:
rows = []
for s_idx, e_idx in event_ranges:
    peak_au12 = float(au12[s_idx:e_idx+1].max())
    peak_au06 = float(au06[s_idx:e_idx+1].max())
    mean_au06 = float(au06[s_idx:e_idx+1].mean())
    start_t = float(timestamps[s_idx])
    end_t = float(timestamps[e_idx])
    duration = end_t - start_t + frame_duration_s  # include last frame's duration
    smile_type = "duchenne" if peak_au06 > AU06_THRESHOLD else "non-duchenne"
    rows.append({
        "start_frame": int(frames[s_idx]),
        "end_frame": int(frames[e_idx]),
        "start_time": round(start_t, 2),
        "end_time": round(end_t, 2),
        "duration_s": round(duration, 2),
        "peak_AU12": round(peak_au12, 3),
        "peak_AU06": round(peak_au06, 3),
        "mean_AU06": round(mean_au06, 3),
        "type": smile_type,
    })

events_df = pd.DataFrame(rows)
events_df

## 3. Overall counts and Duchenne ratio

**What this section does:** reduces the event table to the single summary numbers you'd report in a study: total smiles, how many were genuine vs social, the ratio, and total time in each state.

**Key number — `duchenne_ratio`:** fraction of detected smiles that were Duchenne. Trust literature generally links higher ratios to higher perceived warmth and trustworthiness.

**Interpretation:**
- Ratio close to 1.0 → most smiles were genuine. Typical of relaxed, enjoyable interactions.
- Ratio close to 0.0 → most smiles were polite/social. Typical of professional or strained interactions.
- Few smiles overall → the ratio is unreliable. Report raw counts alongside it.

In [ ]:
n_total = len(events_df)
n_duchenne = int((events_df["type"] == "duchenne").sum()) if n_total else 0
n_non = int((events_df["type"] == "non-duchenne").sum()) if n_total else 0
total_smile_s = float(events_df["duration_s"].sum()) if n_total else 0.0
duchenne_s = float(events_df.loc[events_df["type"] == "duchenne", "duration_s"].sum()) if n_total else 0.0
non_s = total_smile_s - duchenne_s

summary = pd.Series({
    "n_smiles_total": n_total,
    "n_duchenne": n_duchenne,
    "n_non_duchenne": n_non,
    "duchenne_ratio": round(n_duchenne / n_total, 3) if n_total else float("nan"),
    "total_smile_time_s": round(total_smile_s, 2),
    "duchenne_time_s": round(duchenne_s, 2),
    "non_duchenne_time_s": round(non_s, 2),
    "duchenne_time_ratio": round(duchenne_s / total_smile_s, 3) if total_smile_s else float("nan"),
})
summary

## 4. Overlaid AU12/AU06 plot with event shading

**What this chart shows:**
- Green line: AU12 (smile muscle) intensity over time.
- Orange line: AU06 (cheek raiser) intensity over time.
- Horizontal dashed line: the threshold (same for both AUs by default).
- Shaded regions: detected smile events. Green = Duchenne, red = non-Duchenne.

**What to look for:**
- Green-shaded regions where both lines rise together → clear genuine smiles.
- Red-shaded regions where AU12 rises but AU06 stays low → social smiles.
- Unshaded AU06 peaks (without AU12) → not smiles (AU06 also activates during squinting or intense concentration).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(timestamps, au12, color="tab:green", label="AU12 (smile)", lw=1.5)
ax.plot(timestamps, au06, color="tab:orange", label="AU06 (cheek raiser)", lw=1.5)
ax.axhline(AU12_THRESHOLD, color="k", lw=0.6, ls="--", alpha=0.5, label=f"threshold ({AU12_THRESHOLD})")

for _, ev in events_df.iterrows():
    color = "tab:green" if ev["type"] == "duchenne" else "tab:red"
    ax.axvspan(ev["start_time"], ev["end_time"] + frame_duration_s, color=color, alpha=0.15)

ax.set_xlabel("time (s)")
ax.set_ylabel("AU intensity")
ax.set_ylim(0, 1)
ax.set_xlim(timestamps.min(), timestamps.max())
ax.set_title("AU12 and AU06 over time — green shading = Duchenne, red = non-Duchenne")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 5. Save event table for reuse

In [ ]:
events_df.to_parquet(EVENTS_OUT, index=False)
print(f"Saved: {EVENTS_OUT}  ({len(events_df)} events)")